# Valutazioni finali di stage – analisi per il management

Notebook riproducibile dall'inizio alla fine. Ogni step è separato da un'intestazione;
i trattamenti sui dati vengono applicati **solo dopo conferma esplicita** e sono documentati
nel *Registro assunzioni e limiti* in fondo a ogni step.

**Input:** `data/INT_VAL_ASSESSMENT_SET.csv` (non versionato nel repo)
**Output:** tabelle in `outputs/tables/`, grafici in `outputs/figures/`

## Step 0 – Struttura del file (nessuna analisi, nessun trattamento)

**Approccio.** Il file viene letto con *tutte le colonne come testo* (`dtype=str`, `keep_default_na=False`).

*Alternative considerate:*
- lettura con i default di pandas: scartata, perché converte in silenzio `"N/A"` e le celle vuote in `NaN`
  e assegna tipi diversi a colonne omogenee (es. `Listening` diventa testo a causa di un solo valore `BBBB`),
  nascondendo proprio le anomalie che vogliamo vedere;
- lettura dall'`.xlsx`: usato solo come controllo incrociato (deve essere identico al CSV).

Così ogni valore viene visto esattamente come scritto nel file; tipi e anomalie sono *diagnosticati*, non corretti.

In [1]:
# --- Import e percorsi -------------------------------------------------------
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

DATA_CSV  = Path("data/INT_VAL_ASSESSMENT_SET.csv")
DATA_XLSX = Path("data/INT_VAL_ASSESSMENT_SET.xlsx")   # opzionale, solo per controllo incrociato
OUT_TAB   = Path("outputs/tables"); OUT_TAB.mkdir(parents=True, exist_ok=True)

# Colonne attese secondo la traccia (da verificare, non da dare per scontate)
SKILLS = ["Digital", "Teamwork", "Entrepreneurial Mindset", "Critical Thinking", "Creativity",
          "Planning", "Flexibility", "Cultural Awareness", "Writing", "Listening",
          "Time Management", "Responsibility", "Motivation", "Feedback Management",
          "Interpersonal Skills", "Leadership"]
SCORES = SKILLS + ["OVERALL"]                      # 17 colonne su scala attesa 1-10
CATEG  = ["NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO", "Hiring"]
EXPECTED_COLS = ["ID", "DURATA", "NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO"] + SCORES + ["Hiring"]

### 0.1 Controlli sul file grezzo (byte e righe)
Prima di usare pandas guardiamo il file "a mano": codifica, separatore, fine riga, numero di campi per riga.

In [2]:
raw = DATA_CSV.read_bytes()

# Codifica: se non è UTF-8 valido, cerchiamo i byte non ASCII per capire quale codifica usare
try:
    raw.decode("utf-8"); ENCODING = "utf-8"
except UnicodeDecodeError:
    ENCODING = "cp1252"   # nessun byte 0x80-0x9F -> cp1252 e latin-1 coincidono su questo file
non_ascii = Counter(b for b in raw if b >= 0x80)
print("Codifica usata:", ENCODING, "| byte non ASCII:", {hex(k): v for k, v in non_ascii.items()})
print("Esempio decodificato:", bytes([0xE0]).decode(ENCODING))

text  = raw.decode(ENCODING)
lines = [l for l in text.split("\r\n") if l != ""]
print("Fine riga CRLF:", "\r\n" in text)
print("Righe non vuote (incluso header):", len(lines))
print("Numero di ';' per riga:", Counter(l.count(";") for l in lines))

# Controlli automatici sulla struttura grezza
assert all(l.count(";") == 23 for l in lines), "righe con numero di campi diverso da 24"
N_ROWS_RAW = len(lines) - 1          # righe dati, header escluso

Codifica usata: cp1252 | byte non ASCII: {'0xe0': 634}
Esempio decodificato: à
Fine riga CRLF: True
Righe non vuote (incluso header): 4383
Numero di ';' per riga: Counter({23: 4383})


### 0.2 Caricamento (tutto come testo) e verifica delle colonne

In [3]:
df_raw = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING, dtype=str, keep_default_na=False)
print("Shape:", df_raw.shape)

# Verifica indipendente: righe lette da pandas == righe contate sul file grezzo
assert len(df_raw) == N_ROWS_RAW, (len(df_raw), N_ROWS_RAW)
# Verifica colonne attese: stessi nomi, stesso ordine, nessuno spazio spurio
assert list(df_raw.columns) == EXPECTED_COLS, set(df_raw.columns) ^ set(EXPECTED_COLS)
assert all(c == c.strip() for c in df_raw.columns)
print("Colonne verificate:", len(df_raw.columns))

Shape: (4382, 24)
Colonne verificate: 24


### 0.3 Profilo delle colonne
Per ogni colonna: valori distinti, celle vuote, `N/A` letterali, quanti valori sono numeri interi,
token non numerici, min/max dei valori numerici e (per i punteggi) quanti cadono fuori dalla scala 1-10.
Il "tipo diagnosticato" è quello che la colonna *dovrebbe* avere; il "tipo pandas default" mostra cosa
succederebbe con una lettura ingenua.

In [4]:
df_default = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING)   # solo per confronto dei tipi

rows = []
for c in df_raw.columns:
    s   = df_raw[c]
    num = pd.to_numeric(s, errors="coerce")                         # NaN dove non numerico
    tok = s[num.isna() & (s != "") & (s != "N/A")]                  # token anomali (né vuoto né N/A)
    is_num_col = num.notna().mean() > 0.9
    rows.append({
        "colonna": c,
        "tipo_pandas_default": str(df_default[c].dtype),
        "tipo_diagnosticato": (("intero, atteso 1-10" if c in SCORES else "intero") if is_num_col else "categorica (testo)"),
        "n_righe": len(s),
        "n_distinti": s.nunique(),
        "n_vuoti": int((s == "").sum()),
        "n_NA_letterale": int((s == "N/A").sum()),
        "n_numerici": int(num.notna().sum()),
        "token_anomali": {k: int(v) for k, v in tok.value_counts().items()} if is_num_col and len(tok) else "",
        "min": num.min() if is_num_col else "",
        "max": num.max() if is_num_col else "",
        "n_fuori_1_10": int(((num < 1) | (num > 10)).sum()) if c in SCORES else "",
        "n_spazi_iniziali_finali": int((s != s.str.strip()).sum()),
    })
profilo = pd.DataFrame(rows)
profilo.to_csv(OUT_TAB / "step0_profilo_colonne.csv", index=False)
profilo

,colonna,tipo_pandas_default,tipo_diagnosticato,n_righe,n_distinti,n_vuoti,n_NA_letterale,n_numerici,token_anomali,min,max,n_fuori_1_10,n_spazi_iniziali_finali
0,ID,int64,intero,4382,4382,0,0,4382,,1,4388,,0
1,DURATA,int64,intero,4382,247,0,0,4382,,20,45534,,0
2,NAZIONE,str,categorica (testo),4382,65,3,0,0,,,,,0
3,SETTORE AZIENDALE,str,categorica (testo),4382,38,0,0,0,,,,,131
4,AREA FUNZIONALE,str,categorica (testo),4382,23,1,0,0,,,,,0
5,CORSO,str,categorica (testo),4382,38,1,0,0,,,,,0
6,Digital,float64,"intero, atteso 1-10",4382,12,1,65,4316,,1.0,10.0,0,0
7,Teamwork,float64,"intero, atteso 1-10",4382,12,1,90,4291,,1.0,10.0,0,0
8,Entrepreneurial Mindset,float64,"intero, atteso 1-10",4382,13,1,153,4228,,1.0,7000.0,1,0
9,Critical Thinking,float64,"intero, atteso 1-10",4382,12,1,127,4254,,1.0,10.0,0,0


### 0.4 Variabili categoriche: valori distinti e conteggi
I valori sono mostrati con `repr()` per rendere visibili celle vuote (`''`) e spazi finali.

In [5]:
vc_list = []
for c in CATEG:
    vc = df_raw[c].value_counts(dropna=False)
    vc_list.append(pd.DataFrame({"variabile": c, "valore": vc.index, "n": vc.values,
                                 "pct": (vc.values / len(df_raw) * 100).round(2)}))
    # Verifica: la somma dei conteggi deve dare il totale righe
    assert vc.sum() == len(df_raw)
valori_cat = pd.concat(vc_list, ignore_index=True)
valori_cat.to_csv(OUT_TAB / "step0_valori_categoriche.csv", index=False)

for c in CATEG:
    sub = valori_cat[valori_cat.variabile == c]
    print(f"\n=== {c}: {len(sub)} valori distinti (n totale = {sub.n.sum()}) ===")
    print(sub.assign(valore=sub.valore.map(repr))[["valore", "n", "pct"]].to_string(index=False))


=== NAZIONE: 65 valori distinti (n totale = 4382) ===
                            valore    n   pct
                          'ITALIA' 3403 77.66
                         'FRANCIA'  157  3.58
                        'GERMANIA'  114  2.60
                          'BELGIO'   87  1.99
'GRAN BRETAGNA E IRLANDA DEL NORD'   81  1.85
                     'LUSSEMBURGO'   63  1.44
                        'SVIZZERA'   63  1.44
           "STATI UNITI D'AMERICA"   48  1.10
        'CINA REPUBBLICA POPOLARE'   36  0.82
                          'SPAGNA'   33  0.75
                         'TURCHIA'   26  0.59
                     'PAESI BASSI'   25  0.57
           'REPUBBLICA DI POLONIA'   21  0.48
                         'ROMANIA'   18  0.41
                       'SINGAPORE'   17  0.39
             'EMIRATI ARABI UNITI'   16  0.37
                          'GRECIA'   14  0.32
                       'DANIMARCA'   10  0.23
                         'MAROCCO'   10  0.23
                      'PO

### 0.5 Hiring – valori presenti (con verifica indipendente)

In [6]:
hiring_vc = df_raw["Hiring"].value_counts()
print(hiring_vc.rename(index=repr))

# Verifica indipendente: filtro semplice vs value_counts
for v in hiring_vc.index:
    assert (df_raw["Hiring"] == v).sum() == hiring_vc[v]
print("\nRighe con Hiring diverso da SI/NO:")
print(df_raw.loc[~df_raw["Hiring"].isin(["SI", "NO"]), ["ID", "OVERALL", "Hiring"]])

Hiring
'SI'      2494
'NO'      1885
''           2
'Null'       1
Name: count, dtype: int64

Righe con Hiring diverso da SI/NO:
        ID OVERALL Hiring
97      98      10   Null
813    816               
3852  3859      10       


### 0.6 ID e DURATA

In [7]:
ids = df_raw["ID"].astype(int)        # sicuro: verificato in 0.3 che è sempre intero
mancanti = sorted(set(range(1, ids.max() + 1)) - set(ids))
print(f"ID: n={len(ids)}, distinti={ids.nunique()}, min={ids.min()}, max={ids.max()}, "
      f"ordinati={ids.is_monotonic_increasing}, buchi nella sequenza={mancanti}")
assert ids.is_unique

dur = df_raw["DURATA"].astype(int)
print("\nDURATA (giorni) – statistiche descrittive, n =", dur.notna().sum())
print(dur.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1).to_string())

# Valori molto grandi: sono plausibili come seriali di data Excel? (solo diagnosi, nessuna correzione)
grandi = df_raw.loc[dur > 400, ["ID", "DURATA", "NAZIONE", "CORSO"]].copy()
grandi["se_fosse_data_excel"] = pd.to_datetime(grandi["DURATA"].astype(int), unit="D",
                                               origin="1899-12-30").dt.date
grandi

ID: n=4382, distinti=4382, min=1, max=4388, ordinati=True, buchi nella sequenza=[266, 378, 1918, 2500, 3093, 3443]

DURATA (giorni) – statistiche descrittive, n = 4382
count     4382.0
mean       179.9
std       1812.5
min         20.0
1%          30.0
5%          39.0
25%         60.0
50%         91.0
75%        171.0
95%        182.0
99%        344.8
max      45534.0


,ID,DURATA,NAZIONE,CORSO,se_fosse_data_excel
17,18,547,SERBIA,B12,1901-06-30
106,107,45303,FRANCIA,B03,2024-01-12
507,510,45394,ITALIA,B16,2024-04-12
940,943,45485,ITALIA,B01,2024-07-12
1079,1082,45504,ITALIA,B01,2024-07-31
1282,1285,45492,ITALIA,B10,2024-07-19
1891,1894,45527,ITALIA,B11,2024-08-23
2302,2306,729,ITALIA,B17,1901-12-29
3139,3145,45534,GRAN BRETAGNA E IRLANDA DEL NORD,T07,2024-08-30


### 0.7 Anomalie puntuali e duplicati (solo elenco, nessuna correzione)

In [8]:
num_scores = df_raw[SCORES].apply(pd.to_numeric, errors="coerce")

# Righe con almeno una cella vuota
righe_vuote = df_raw[(df_raw == "").any(axis=1)]
print(f"Righe con almeno una cella vuota: n={len(righe_vuote)}")
print(righe_vuote[["ID", "DURATA", "NAZIONE", "AREA FUNZIONALE", "CORSO", "OVERALL", "Hiring"]], "\n")

# Punteggi fuori scala (numerici fuori 1-10) o token non numerici diversi da N/A
fuori = (num_scores < 1) | (num_scores > 10)
token = num_scores.isna() & ~df_raw[SCORES].isin(["", "N/A"])
anom = df_raw.loc[(fuori | token).any(axis=1), ["ID"] + SCORES]
print(f"Righe con punteggi fuori scala o token anomali: n={len(anom)}")
print(anom.to_string(index=False), "\n")

# N/A letterali per riga
na_per_riga = (df_raw[SCORES] == "N/A").sum(axis=1)
print("Distribuzione n. di N/A per riga:", na_per_riga.value_counts().sort_index().to_dict())
print("Righe con almeno un N/A:", int((na_per_riga > 0).sum()), "\n")

# Duplicati: identici su tutte le colonne tranne ID
dup_mask = df_raw.drop(columns="ID").duplicated(keep=False)
print(f"Duplicati esatti (escluso ID): righe coinvolte={dup_mask.sum()}, "
      f"copie in eccesso={df_raw.drop(columns='ID').duplicated().sum()}")
print("di cui con tutti i 17 punteggi = 10:", int((df_raw.loc[dup_mask, SCORES] == "10").all(axis=1).sum()))
print("Righe totali con tutti i 17 punteggi = 10:", int((df_raw[SCORES] == "10").all(axis=1).sum()))

pd.concat([
    anom.assign(tipo="punteggio fuori scala / token")[["ID", "tipo"]],
    righe_vuote.assign(tipo="cella vuota")[["ID", "tipo"]],
    df_raw.loc[dup_mask].assign(tipo="duplicato escluso ID")[["ID", "tipo"]],
    df_raw.loc[dur > 400].assign(tipo="DURATA > 400")[["ID", "tipo"]],
]).sort_values("ID").to_csv(OUT_TAB / "step0_righe_anomale.csv", index=False)

Righe con almeno una cella vuota: n=4
        ID DURATA NAZIONE                       AREA FUNZIONALE CORSO OVERALL Hiring
813    816    330                                                                   
1131  1134    212                            Ricerca & Sviluppo   B01      10     NO
2027  2031     85                       Attività polifunzionali   T01      10     SI
3852  3859    195  ITALIA  Amministrazione, Finanza e Controllo   M03      10        

Righe con punteggi fuori scala o token anomali: n=8
  ID Digital Teamwork Entrepreneurial Mindset Critical Thinking Creativity Planning Flexibility Cultural Awareness Writing Listening Time Management Responsibility Motivation Feedback Management Interpersonal Skills Leadership OVERALL
   1      10       10                      10                10         10       10          10                  A      10        10              10             10         10                  10                   10         10      10
  42      10 

### 0.8 Verifica indipendente: CSV vs XLSX

In [9]:
if DATA_XLSX.exists():
    xl = pd.read_excel(DATA_XLSX, sheet_name=None, dtype=str, keep_default_na=False)
    print("Fogli:", {k: v.shape for k, v in xl.items()})
    x = next(iter(xl.values()))
    assert list(x.columns) == list(df_raw.columns) and x.shape == df_raw.shape
    n_diff = int((x.values != df_raw.values).sum())
    print("Celle diverse tra CSV e XLSX:", n_diff)
    assert n_diff == 0
else:
    print("XLSX non presente: controllo saltato")

Fogli: {'in': (4382, 24)}
Celle diverse tra CSV e XLSX: 0


### Registro assunzioni e limiti – aggiornato a fine Step 0

| # | Tipo | Voce | Stato |
|---|------|------|-------|
| A1 | Assunzione | Codifica `cp1252` (unico byte non ASCII: `0xE0` = "à"); separatore `;`; fine riga CRLF. | verificata |
| A2 | Assunzione | `NAZIONE` = paese della sede dello stage, non nazionalità dello studente (da brief). | da brief |
| A3 | Assunzione | `CORSO` trattato solo come codice anonimo; nessuna ipotesi sul tipo di corso. | da brief |
| A4 | Assunzione | Scala attesa dei punteggi: interi 1-10. | da brief, coerente con 99,9% dei valori |
| L1 | Qualità | `N/A` letterale in 15 skill su 16 (non in `Leadership` né in `OVERALL`); 470 righe con almeno un N/A. Significato (non osservato? non applicabile?) ignoto. | da decidere |
| L2 | Qualità | 7 valori fuori scala (7000, 88, 999, 55, 100) e 3 token non numerici (`A`, `BBBB`, `NN`). | da decidere |
| L3 | Qualità | Righe con celle vuote: ID 816 (quasi vuota), 1134 e 2031 (NAZIONE vuota), 3859 (Hiring vuoto); ID 98 ha Hiring = `Null`. | da decidere |
| L4 | Qualità | `DURATA`: 7 valori tra 45.303 e 45.534 compatibili con date Excel 2024; 2 valori (547, 729) alti ma non impossibili. | da decidere |
| L5 | Qualità | 9 coppie identiche su tutte le colonne tranne ID; 8 su 9 sono valutazioni "tutto 10", che nel file sono 742: potrebbero essere coincidenze. | da decidere |
| L6 | Qualità | `SETTORE AZIENDALE` 'ONG ed Enti No Profit ' ha uno spazio finale (unica categoria, nessuno sdoppiamento). | da decidere (cosmetico) |
| L7 | Qualità | `CORSO` = 'L' (n=314) ha formato diverso dagli altri codici (lettera + 2 cifre). | da chiarire |
| L8 | Limite | Forte sbilanciamento: ITALIA 3.403 righe su 4.382; molti paesi, corsi e aree con n molto piccolo. | da gestire in analisi |
| L9 | Limite | ID non contigui (mancano 6 ID): possibile pre-filtraggio a monte, ma nessun duplicato di ID. | informativo |

## Step 1 – Audit di qualità del dato (solo diagnosi, nessun trattamento)

**Regola dello step:** `df_raw` non viene mai modificato. Tutti i calcoli usano *viste temporanee*
(maschere booleane e conversioni numeriche in variabili separate). A fine step un'impronta (hash)
del dataset verifica che sia identico a quello letto nello Step 0.

**Approccio e alternative considerate**

| Controllo | Scelta | Alternativa scartata e perché |
|---|---|---|
| Classificazione delle celle dei voti | Ogni cella in **una sola** categoria: valido 1-10, `N/A`, vuoto, testo non numerico, decimale, fuori scala (con assert di partizione) | Conversione diretta a numerico: confonde `N/A`, vuoti e refusi testuali in un unico `NaN` |
| Duplicati | Per ID e per riga intera escluso ID | Duplicati "quasi uguali" (fuzzy): senza un identificativo dello studente non sono distinguibili da stage diversi |
| Etichette | Normalizzazione (maiuscole, accenti, punteggiatura) + similarità testuale tra coppie, poi lettura a occhio | Correzione automatica delle varianti: rischia di unire categorie diverse |
| DURATA | Soglie esplicite: < 30 gg, > 365 gg, seriali Excel 2000-2030; regola IQR solo come riferimento | Solo regola IQR: la distribuzione è asimmetrica e segnalerebbe come anomali stage normali di 6 mesi |
| Mancanti casuali? | Confronto del tasso di Hiring (e di OVERALL, DURATA, segmenti) tra chi ha e chi non ha skill mancanti; Fisher/chi-quadro, correzione di Holm per i test per singola skill | Test di Little (MCAR): non disponibile nelle librerie concordate e assume normalità multivariata, inadatta a voti 1-10 molto asimmetrici |
| OVERALL vs skill | Spearman (principale, voti ordinali e asimmetrici) e Pearson; scarto OVERALL − media skill; ripetuto su 3 campioni | Solo Pearson: gonfiato dalle molte valutazioni "tutto 10" |
| Valutazioni piatte | 16 skill tutte valide e identiche; separatamente le piatte con qualche `N/A` | Deviazione standard ≤ soglia: soglia arbitraria |

Nota: un test può solo **trovare evidenza contro** la casualità dei mancanti, mai dimostrarla.

In [10]:
# --- Step 1: setup dell'audit (nessuna modifica a df_raw) -------------------------------
import re, unicodedata, difflib
from itertools import combinations
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

# Impronta del dataset grezzo: a fine step verifichiamo che sia identica (nessun trattamento applicato)
FINGERPRINT_RAW = int(pd.util.hash_pandas_object(df_raw, index=True).sum())

N_TOT    = len(df_raw)
LABELS   = ["NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO"]
SOGLIA_N = 30                      # numerosità minima concordata per mostrare un segmento

def pct(n, den=N_TOT):
    """Percentuale con 2 decimali (NaN se il denominatore è 0)."""
    return round(100 * n / den, 2) if den else np.nan

def n_pct(n, den=N_TOT):
    """Formato 'n (x% di den)' per riportare sempre la numerosità."""
    return f"{int(n)} ({pct(n, den):.2f}% di {int(den)})"

audit = []   # righe della tabella riassuntiva, riempita sezione per sezione
def registra(problema, colonne, mask_righe, trattamento, rischio, decisione):
    """Aggiunge un problema alla tabella riassuntiva; n e % sono calcolati dalla maschera di righe."""
    n = int(np.asarray(mask_righe).sum())
    audit.append({"problema": problema, "colonne": colonne, "n_righe": n, "pct_sul_totale": pct(n),
                  "trattamento_proposto": trattamento, "rischio_del_trattamento": rischio,
                  "decisione": decisione})

### 1.1 Skill e OVERALL: valori non numerici, mancanti, fuori scala

In [11]:
txt = df_raw[SCORES]                                    # testo originale
num = txt.apply(pd.to_numeric, errors="coerce")         # vista numerica temporanea (NaN se non numero)

m_na     = txt.eq("N/A")                                # mancante dichiarato
m_vuoto  = txt.eq("")                                   # cella vuota
m_token  = num.isna() & ~m_na & ~m_vuoto                # testo non numerico (es. 'A', 'BBBB')
in_scala = (num >= 1) & (num <= 10)
m_fuori  = num.notna() & ~in_scala                      # numero fuori dalla scala 1-10
m_dec    = in_scala & (num % 1 != 0)                    # numero nella scala ma con decimali
m_valido = in_scala & (num % 1 == 0)                    # voto valido

# Controllo automatico: ogni cella appartiene a una e una sola categoria
partizione = sum(m.astype(int) for m in [m_na, m_vuoto, m_token, m_fuori, m_dec, m_valido])
assert (partizione == 1).all().all(), "classificazione delle celle non esclusiva"

def elenca(mask, col):
    """Elenco 'ID: valore' delle celle segnalate in una colonna."""
    return "; ".join(f"ID {i}: {v!r}" for i, v in zip(df_raw.loc[mask[col], "ID"], txt.loc[mask[col], col]))

qual_voti = pd.DataFrame({
    "n_validi": m_valido.sum(), "n_NA": m_na.sum(), "n_vuoti": m_vuoto.sum(),
    "n_testo_non_numerico": m_token.sum(), "n_decimali": m_dec.sum(), "n_fuori_scala": m_fuori.sum(),
})
qual_voti["pct_non_validi"] = (100 * (N_TOT - qual_voti["n_validi"]) / N_TOT).round(2)
qual_voti["celle_anomale"] = [elenca(m_token | m_fuori, c) for c in SCORES]
qual_voti.loc["TOTALE (celle)"] = list(qual_voti.iloc[:, :6].sum()) + [np.nan, ""]
qual_voti.to_csv(OUT_TAB / "step1_qualita_voti.csv")
display(qual_voti)

# Livello riga
r_token = m_token.any(axis=1)
r_fuori = m_fuori.any(axis=1)
r_na    = m_na.any(axis=1)
r_vuota = df_raw.drop(columns="ID").eq("").sum(axis=1) >= 10      # riga (quasi) vuota
print("Righe con testo non numerico:", n_pct(r_token.sum()))
print("Righe con valori fuori scala:", n_pct(r_fuori.sum()))
print("Righe con almeno un N/A    :", n_pct(r_na.sum()), "| celle N/A:", int(m_na.values.sum()))
print("Righe quasi vuote (>=10 celle vuote):", n_pct(r_vuota.sum()), "-> ID", df_raw.loc[r_vuota, "ID"].tolist())
assert m_dec.values.sum() == 0, "presenti voti decimali: da esaminare"

,n_validi,n_NA,n_vuoti,n_testo_non_numerico,n_decimali,n_fuori_scala,pct_non_validi,celle_anomale
Digital,4316,65,1,0,0,0,1.51,
Teamwork,4291,90,1,0,0,0,2.08,
Entrepreneurial Mindset,4227,153,1,0,0,1,3.54,ID 42: '7000'
Critical Thinking,4254,127,1,0,0,0,2.92,
Creativity,4285,96,1,0,0,0,2.21,
Planning,4286,95,1,0,0,0,2.19,
Flexibility,4369,12,1,0,0,0,0.30,
Cultural Awareness,4330,49,1,1,0,1,1.19,ID 1: 'A'; ID 3919: '88'
Writing,4375,5,1,0,0,1,0.16,ID 3863: '999'
Listening,4377,3,1,1,0,0,0.11,ID 4378: 'BBBB'


Righe con testo non numerico: 3 (0.07% di 4382)
Righe con valori fuori scala: 5 (0.11% di 4382)
Righe con almeno un N/A    : 470 (10.73% di 4382) | celle N/A: 772
Righe quasi vuote (>=10 celle vuote): 1 (0.02% di 4382) -> ID ['816']


In [12]:
# Dettaglio dei fuori scala: confronto con le altre valutazioni della stessa riga (solo plausibilità)
# Proposte = decisioni già prese con il committente (Step 0): NON vengono applicate qui.
PROPOSTA_FUORI_SCALA = {"55": "5", "88": "8", "999": "9", "100": "10", "7000": "mancante"}
righe = []
for col in SCORES:
    for idx in df_raw.index[m_fuori[col]]:
        altre = num.loc[idx, SCORES].drop(col).where(m_valido.loc[idx, SCORES].drop(col))
        righe.append({"ID": df_raw.at[idx, "ID"], "colonna": col, "valore": txt.at[idx, col],
                      "altre_valide_n": int(altre.notna().sum()), "altre_min": altre.min(),
                      "altre_mediana": altre.median(), "altre_max": altre.max(),
                      "proposta_concordata": PROPOSTA_FUORI_SCALA.get(txt.at[idx, col], "DA DECIDERE")})
fuori_scala = pd.DataFrame(righe).sort_values("ID")
fuori_scala.to_csv(OUT_TAB / "step1_fuori_scala.csv", index=False)
display(fuori_scala)

ricod = df_raw["ID"].isin(fuori_scala.loc[fuori_scala.proposta_concordata.str.isdigit(), "ID"])
non_ricod = r_fuori & ~ricod

cols = lambda mask: ", ".join(c for c in SCORES if mask[c].any())
registra("Testo non numerico nei voti", cols(m_token), r_token,
         "Convertire in mancante solo per quella skill", "Minimo: 3 celle; il voto reale è irrecuperabile", "confermato")
registra("Fuori scala ricodificabile come refuso (55, 88, 999, 100)", cols(m_fuori & ricod.values[:, None]), ricod,
         "Ricodificare in 5, 8, 9, 10", "Il valore reale potrebbe essere diverso (es. 999 è spesso un codice di 'mancante')", "confermato")
registra("Fuori scala non ricodificabile (7000)", cols(m_fuori & non_ricod.values[:, None]), non_ricod,
         "Convertire in mancante solo per quella skill", "Minimo: 1 cella", "confermato")
registra("Mancante dichiarato (N/A)", f"{int((m_na.sum() > 0).sum())} skill su 16 (non Leadership, non OVERALL)", r_na,
         "Mancante solo per quella skill; nessuna esclusione di riga, nessuna imputazione",
         "Se i mancanti non sono casuali (1.6) le medie per skill possono essere distorte; n diverso tra skill", "confermato")
registra("Riga quasi vuota", "tutte tranne ID, DURATA, SETTORE", r_vuota,
         "Escludere la riga", "Nessuno rilevante: la riga non contiene valutazioni", "confermato")

,ID,colonna,valore,altre_valide_n,altre_min,altre_mediana,altre_max,proposta_concordata
4,3811,OVERALL,100,16,9.0,10.0,10.0,10
2,3863,Writing,999,16,7.0,9.0,10.0,9
1,3919,Cultural Awareness,88,16,5.0,6.0,8.0,8
0,42,Entrepreneurial Mindset,7000,16,10.0,10.0,10.0,mancante
3,73,Responsibility,55,16,4.0,6.0,7.0,5


### 1.2 Duplicati

In [13]:
dup_id  = df_raw["ID"].duplicated(keep=False)
senza_id = df_raw.drop(columns="ID")
dup_riga = senza_id.duplicated(keep=False)

# Verifica indipendente: groupby su tutte le colonne tranne ID
gruppi = df_raw.groupby(list(senza_id.columns), sort=False)["ID"].agg(list)
gruppi = gruppi[gruppi.str.len() > 1]
assert gruppi.str.len().sum() == dup_riga.sum()

coppie = pd.DataFrame({
    "ID": [", ".join(g) for g in gruppi.values],
    "voto_unico_se_piatta": [k[5] if len(set(k[5:22])) == 1 else "-" for k in gruppi.index],  # 0-4 metadati, 5-21 voti, 22 Hiring
    "DURATA": [k[0] for k in gruppi.index], "NAZIONE": [k[1] for k in gruppi.index],
    "CORSO": [k[4] for k in gruppi.index], "Hiring": [k[-1] for k in gruppi.index],
})
coppie.to_csv(OUT_TAB / "step1_righe_identiche.csv", index=False)
print("Duplicati di ID:", n_pct(dup_id.sum()))
print("Righe identiche escluso ID:", n_pct(dup_riga.sum()), f"in {len(gruppi)} gruppi")
display(coppie)

registra("ID duplicato", "ID", dup_id, "Nessuno", "-", "nessun caso")
registra("Righe identiche su tutte le colonne tranne ID", "tutte tranne ID", dup_riga,
         "Nessuno: ID diverso = stage diverso", "Se fossero doppi inserimenti, 9 valutazioni (quasi tutte 'tutto 10') conterebbero due volte",
         "confermato")

Duplicati di ID: 0 (0.00% di 4382)
Righe identiche escluso ID: 18 (0.41% di 4382) in 9 gruppi


,ID,voto_unico_se_piatta,DURATA,NAZIONE,CORSO,Hiring
0,"216, 954",10,179,ITALIA,B01,SI
1,"865, 916",9,95,ITALIA,T01,SI
2,"1052, 1369",10,178,FRANCIA,B04,SI
3,"1065, 1211",10,90,ITALIA,T08,SI
4,"1381, 2388",10,179,ITALIA,B03,SI
5,"1786, 4161",10,39,ITALIA,T01,SI
6,"2208, 2714",10,53,ITALIA,B14,SI
7,"3217, 3602",10,41,ITALIA,T01,SI
8,"3391, 3397",10,109,ITALIA,B11,SI


### 1.3 Etichette di NAZIONE, SETTORE AZIENDALE, AREA FUNZIONALE, CORSO

In [14]:
def normalizza(s):
    """Chiave di confronto: maiuscolo, senza accenti, solo lettere e cifre."""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Z0-9]", "", s.upper())

sintesi_lab, simili = [], []
for c in LABELS:
    s = df_raw[c]
    vc = s[s != ""].value_counts()
    vals = pd.Series(vc.index)
    chiavi = vals.map(normalizza)
    sintesi_lab.append({
        "colonna": c, "n_etichette": len(vals),
        "n_righe_mancanti": int((s == "").sum()),
        "n_righe_spazi_extra": int((s != s.str.strip()).sum()),
        "etichette_con_spazi": [v for v in vals if v != v.strip()],
        "varianti_stessa_chiave": sorted(vals[chiavi.duplicated(keep=False)].tolist()),
        "n_categorie_n_lt_30": int((vc < SOGLIA_N).sum()),
        "n_righe_in_categorie_n_lt_30": int(vc[vc < SOGLIA_N].sum()),
    })
    # coppie di etichette molto simili (da leggere a occhio: simili non significa uguali)
    for a, b in combinations(vals, 2):
        r = difflib.SequenceMatcher(None, normalizza(a), normalizza(b)).ratio()
        if r >= 0.85:
            simili.append({"colonna": c, "etichetta_1": a, "n_1": int(vc[a]),
                           "etichetta_2": b, "n_2": int(vc[b]), "similarita": round(r, 2)})
sintesi_lab = pd.DataFrame(sintesi_lab)
simili = pd.DataFrame(simili, columns=["colonna", "etichetta_1", "n_1", "etichetta_2", "n_2", "similarita"])
sintesi_lab.to_csv(OUT_TAB / "step1_etichette_sintesi.csv", index=False)
simili.to_csv(OUT_TAB / "step1_etichette_simili.csv", index=False)
display(sintesi_lab)
print("Coppie di etichette con similarità >= 0,85:")
display(simili)

# Etichette residuali / generiche (poco interpretabili)
PATTERN_RESIDUALI = r"^(?:Altro|Altri|Altre|Generico)\b|non in elenco|polifunzional"
residuali = {c: df_raw[c][df_raw[c].str.contains(PATTERN_RESIDUALI, regex=True)].value_counts().to_dict()
             for c in ["SETTORE AZIENDALE", "AREA FUNZIONALE"]}
print("Etichette residuali:", residuali)

# Formato dei codici CORSO: atteso lettera + 2 cifre
corso_fuori_formato = df_raw["CORSO"].ne("") & ~df_raw["CORSO"].str.fullmatch(r"[A-Z]\d{2}")
print("CORSO fuori formato:", df_raw.loc[corso_fuori_formato, "CORSO"].value_counts().to_dict())

,colonna,n_etichette,n_righe_mancanti,n_righe_spazi_extra,etichette_con_spazi,varianti_stessa_chiave,n_categorie_n_lt_30,n_righe_in_categorie_n_lt_30
0,NAZIONE,64,3,0,[],[],54,294
1,SETTORE AZIENDALE,38,0,131,[ONG ed Enti No Profit ],[],0,0
2,AREA FUNZIONALE,22,1,0,[],[],7,137
3,CORSO,37,1,0,[],[],13,95


Coppie di etichette con similarità >= 0,85:


,colonna,etichetta_1,n_1,etichetta_2,n_2,similarita
0,NAZIONE,AUSTRIA,6,AUSTRALIA,5,0.88


Etichette residuali: {'SETTORE AZIENDALE': {'Altre industrie manufatturiere': 131, 'Altri enti locali, amministrazioni centrali, authority': 120, 'Altri servizi alle imprese e alla persona': 109, 'Altri studi professionali': 100}, 'AREA FUNZIONALE': {'Attività polifunzionali': 217, 'Legale - Ulteriori aree non in elenco': 47, 'Generico': 18, 'Altro': 3}}
CORSO fuori formato: {'L': 314}


In [15]:
# Plausibilità di SETTORE AZIENDALE
# (a) distribuzione: è compatibile con un'assegnazione uniforme tra i 38 settori?
vc_sett = df_raw["SETTORE AZIENDALE"].value_counts()
chi_unif = stats.chisquare(vc_sett.values)
print(f"SETTORE: {len(vc_sett)} settori, n per settore min={vc_sett.min()} max={vc_sett.max()}; "
      f"test di uniformità chi2={chi_unif.statistic:.1f}, p={chi_unif.pvalue:.3f} (n={vc_sett.sum()})")

# (b) coerenza con l'area funzionale: nei dati reali ci aspetteremmo, ad esempio,
#     più aree legali negli studi legali che negli altri settori
area = df_raw["AREA FUNZIONALE"]; con_area = area.ne("")
sett = df_raw["SETTORE AZIENDALE"].str.strip()
CONTROLLI = [("Studi legali e notarili", area.str.startswith("Legale"), "Legale (tutte le sottovoci)"),
             ("Informatica e attività connesse", area.eq("Sistemi Informativi"), "Sistemi Informativi"),
             ("Revisione contabile", area.eq("Amministrazione, Finanza e Controllo"), "Amministrazione, Finanza e Controllo"),
             ("Studi commercialisti, tributaristi ed esperti contabili", area.eq("Amministrazione, Finanza e Controllo"), "Amministrazione, Finanza e Controllo")]
righe = []
for s_nome, m_area, a_nome in CONTROLLI:
    dentro, fuori = con_area & sett.eq(s_nome), con_area & ~sett.eq(s_nome)
    k_in, n_in, k_out, n_out = (dentro & m_area).sum(), dentro.sum(), (fuori & m_area).sum(), fuori.sum()
    p = stats.fisher_exact([[k_in, n_in - k_in], [k_out, n_out - k_out]])[1]
    righe.append({"settore": s_nome, "area_attesa": a_nome, "n_settore": n_in, "pct_area_nel_settore": pct(k_in, n_in),
                  "n_altri_settori": n_out, "pct_area_negli_altri": pct(k_out, n_out), "p_fisher": round(p, 3)})
coerenza_settore = pd.DataFrame(righe)
coerenza_settore.to_csv(OUT_TAB / "step1_coerenza_settore_area.csv", index=False)
display(coerenza_settore)
registra(f"SETTORE AZIENDALE senza struttura plausibile (distribuzione uniforme p={chi_unif.pvalue:.2f}; nessuna coerenza con l'area)",
         "SETTORE AZIENDALE", pd.Series(True, index=df_raw.index),
         "Non usarlo come dimensione di segmentazione; al più descrittivo, dichiarando il motivo",
         "Se il dato fosse corretto, si rinuncerebbe a una dimensione di lettura", "DA DECIDERE")

SETTORE: 38 settori, n per settore min=100 max=139; test di uniformità chi2=28.3, p=0.848 (n=4382)


,settore,area_attesa,n_settore,pct_area_nel_settore,n_altri_settori,pct_area_negli_altri,p_fisher
0,Studi legali e notarili,Legale (tutte le sottovoci),113,7.08,4268,8.13,0.861
1,Informatica e attività connesse,Sistemi Informativi,106,2.83,4275,3.13,1.000
2,Revisione contabile,"Amministrazione, Finanza e Controllo",103,44.66,4278,39.20,0.263
3,"Studi commercialisti, tributaristi ed esperti ...","Amministrazione, Finanza e Controllo",125,34.40,4256,39.47,0.266


In [16]:
# Registrazione dei problemi di etichetta
naz, ar, co = df_raw["NAZIONE"], df_raw["AREA FUNZIONALE"], df_raw["CORSO"]
vc_naz = naz[naz != ""].value_counts(); vc_co = co[co != ""].value_counts()
area_gr = ar.where(~ar.str.startswith("Legale"), "Legale")             # raggruppamento concordato (solo vista)
vc_ar = area_gr[area_gr != ""].value_counts()

registra("NAZIONE mancante", "NAZIONE", naz.eq(""), "Mancante solo per le analisi per paese",
         "Minimo: 3 righe (una è la riga quasi vuota)", "confermato")
registra("AREA FUNZIONALE o CORSO mancante", "AREA FUNZIONALE, CORSO", ar.eq("") | co.eq(""),
         "Mancante solo per quella variabile", "Nessuno: è la riga quasi vuota", "confermato")
registra("Spazio finale nell'etichetta 'ONG ed Enti No Profit '", "SETTORE AZIENDALE",
         df_raw["SETTORE AZIENDALE"].ne(df_raw["SETTORE AZIENDALE"].str.strip()),
         "Rimuovere gli spazi iniziali/finali", "Nessuno (cosmetico, nessuna categoria si fonde)", "DA DECIDERE")
registra("Codice CORSO con formato diverso ('L')", "CORSO", corso_fuori_formato,
         "Nessuno: trattato come codice valido", "Se fosse un codice troncato, unirebbe corsi diversi", "confermato")
registra("Etichette residuali/generiche in AREA FUNZIONALE", ", ".join(residuali["AREA FUNZIONALE"]),
         ar.isin(residuali["AREA FUNZIONALE"]),
         "Tenere; leggerle come non interpretabili; nei confronti solo se n>=30",
         "Contenuto eterogeneo: un risultato su queste aree non indica un'area precisa", "DA DECIDERE")
registra("Paesi con n<30 (long tail)", f"NAZIONE ({int((vc_naz < SOGLIA_N).sum())} paesi)", naz.isin(vc_naz[vc_naz < SOGLIA_N].index),
         "Raggruppare in 'Altri esteri' (opzione B)", "Gruppo eterogeneo: la media nasconde differenze tra paesi", "confermato")
registra("Corsi con n<30", f"CORSO ({int((vc_co < SOGLIA_N).sum())} codici)", co.isin(vc_co[vc_co < SOGLIA_N].index),
         "Nessun accorpamento; esclusi dai confronti tra corsi, inclusi nei totali", "Nessuna informazione per questi corsi", "confermato")
registra("Aree con n<30 (dopo raggruppamento 'Legale')", ", ".join(vc_ar[vc_ar < SOGLIA_N].index), area_gr.isin(vc_ar[vc_ar < SOGLIA_N].index),
         "Escluse dai confronti tra aree, incluse nei totali", "Nessuna informazione per queste aree", "confermato")

### 1.4 DURATA: valori implausibili e possibili date Excel

In [17]:
dur = pd.to_numeric(df_raw["DURATA"], errors="coerce")
assert dur.notna().all() and (dur > 0).all()

ORIGINE_EXCEL = pd.Timestamp("1899-12-30")                       # origine dei seriali di data Excel
SERIALE_MIN = (pd.Timestamp("2000-01-01") - ORIGINE_EXCEL).days   # 36526
SERIALE_MAX = (pd.Timestamp("2030-12-31") - ORIGINE_EXCEL).days
m_excel = dur.between(SERIALE_MIN, SERIALE_MAX)                    # compatibile con una data 2000-2030
m_alta  = (dur > 365) & ~m_excel                                   # oltre un anno
m_bassa = dur < 30                                                 # meno di un mese

q1, q3 = dur[~m_excel].quantile([.25, .75]); soglia_iqr = q3 + 1.5 * (q3 - q1)
print(f"DURATA senza seriali Excel: n={int((~m_excel).sum())}, mediana={dur[~m_excel].median():.0f}, "
      f"Q1={q1:.0f}, Q3={q3:.0f}, soglia IQR={soglia_iqr:.1f} -> oltre soglia: {n_pct((dur[~m_excel] > soglia_iqr).sum())}")
print("Possibili date Excel:", n_pct(m_excel.sum()))
print("Oltre 365 giorni (non date):", n_pct(m_alta.sum()))
print("Sotto 30 giorni:", n_pct(m_bassa.sum()), "| valori:", dur[m_bassa].value_counts().sort_index().to_dict())

durata_anomala = df_raw.loc[m_excel | m_alta, ["ID", "DURATA", "NAZIONE", "CORSO"]].assign(
    tipo=np.where(m_excel[m_excel | m_alta], "possibile data Excel", "oltre 365 giorni"),
    data_se_seriale_excel=lambda d: np.where(d.tipo.eq("possibile data Excel"),
        (ORIGINE_EXCEL + pd.to_timedelta(d.DURATA.astype(int), unit="D")).dt.date.astype(str), ""))
durata_anomala.to_csv(OUT_TAB / "step1_durata_anomala.csv", index=False)
display(durata_anomala)

registra("DURATA compatibile con una data Excel (2024)", "DURATA", m_excel, "Convertire in mancante",
         "La durata reale è persa; nessuna ricostruzione possibile", "confermato")
registra("DURATA oltre 365 giorni", "DURATA", m_alta, "Tenere e segnalare con un flag",
         "Valori veri ma rari possono pesare sulle medie di durata: usare la mediana", "confermato per 547 e 729; DA DECIDERE per gli altri 7")
registra("DURATA sotto 30 giorni", "DURATA", m_bassa, "Tenere: plausibile per stage brevi",
         "Nessuno", "DA DECIDERE")

DURATA senza seriali Excel: n=4375, mediana=91, Q1=60, Q3=170, soglia IQR=335.0 -> oltre soglia: 39 (0.89% di 4382)
Possibili date Excel: 7 (0.16% di 4382)
Oltre 365 giorni (non date): 9 (0.21% di 4382)
Sotto 30 giorni: 40 (0.91% di 4382) | valori: {20: 1, 23: 2, 24: 1, 25: 28, 27: 2, 28: 4, 29: 2}


,ID,DURATA,NAZIONE,CORSO,tipo,data_se_seriale_excel
17,18,547,SERBIA,B12,oltre 365 giorni,
33,34,374,ITALIA,M13,oltre 365 giorni,
106,107,45303,FRANCIA,B03,possibile data Excel,2024-01-12
122,123,366,GERMANIA,B14,oltre 365 giorni,
128,129,389,ITALIA,T07,oltre 365 giorni,
507,510,45394,ITALIA,B16,possibile data Excel,2024-04-12
786,789,373,ITALIA,B16,oltre 365 giorni,
926,929,368,ITALIA,B01,oltre 365 giorni,
934,937,389,ITALIA,B04,oltre 365 giorni,
940,943,45485,ITALIA,B01,possibile data Excel,2024-07-12


### 1.5 Hiring: valori diversi da SI/NO

In [18]:
hir = df_raw["Hiring"]
m_hir_bad = ~hir.isin(["SI", "NO"])
print("Hiring diverso da SI/NO:", n_pct(m_hir_bad.sum()))
display(df_raw.loc[m_hir_bad, ["ID", "OVERALL", "Hiring"]].assign(Hiring=lambda d: d.Hiring.map(repr)))
registra("Hiring diverso da SI/NO ('' e 'Null')", "Hiring", m_hir_bad, "Mancante solo per le analisi su Hiring",
         "Minimo: 3 righe", "confermato")

Hiring diverso da SI/NO: 3 (0.07% di 4382)


,ID,OVERALL,Hiring
97,98,10,'Null'
813,816,,''
3852,3859,10,''


### 1.6 Mancanti: distribuzione per skill e legame con Hiring (mancanti casuali o no?)
Campione per i confronti con Hiring: righe con Hiring = SI/NO. "Mancante" = `N/A` dichiarato
(i 3 testi non numerici e i fuori scala sono esclusi da questa definizione).

In [19]:
m_na_sk = m_na[SKILLS]                              # OVERALL non contiene N/A
k_miss  = m_na_sk.sum(axis=1)                       # numero di skill mancanti per riga
camp    = hir.isin(["SI", "NO"])
y       = hir.eq("SI")

# (a) distribuzione dei mancanti per skill (denominatore: tutte le righe)
miss_skill = pd.DataFrame({"n_mancanti": m_na_sk.sum(), "pct_mancanti": (100 * m_na_sk.sum() / N_TOT).round(2),
                           "n_totale": N_TOT}).sort_values("n_mancanti", ascending=False)
miss_skill.to_csv(OUT_TAB / "step1_mancanti_per_skill.csv")
display(miss_skill)
print("Righe per numero di skill mancanti:", k_miss.value_counts().sort_index().to_dict())

def tasso(mask):
    """n, n_SI, tasso di Hiring e intervallo di Wilson al 95% sul sottoinsieme mask (dentro il campione)."""
    m = mask & camp
    n, k = int(m.sum()), int((m & y).sum())
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    return {"n": n, "n_SI": k, "tasso_SI_pct": pct(k, n), "IC95_basso": round(100 * lo, 1), "IC95_alto": round(100 * hi, 1)}

# (b) Hiring: chi ha almeno una skill mancante vs nessuna
gruppi_miss = pd.DataFrame({"≥1 skill mancante": tasso(k_miss > 0), "nessuna skill mancante": tasso(k_miss == 0)}).T
display(gruppi_miss)
tab = np.array([[gruppi_miss.n_SI.iloc[0], gruppi_miss.n.iloc[0] - gruppi_miss.n_SI.iloc[0]],
                [gruppi_miss.n_SI.iloc[1], gruppi_miss.n.iloc[1] - gruppi_miss.n_SI.iloc[1]]], dtype=float)
chi2, p_chi2, _, attesi = stats.chi2_contingency(tab, correction=False)
p_fisher = stats.fisher_exact(tab)[1]
# Verifica indipendente: chi-quadro calcolato a mano
attesi_manuali = tab.sum(1, keepdims=True) * tab.sum(0, keepdims=True) / tab.sum()
chi2_manuale = ((tab - attesi_manuali) ** 2 / attesi_manuali).sum()
assert np.isclose(chi2, chi2_manuale)
print(f"Differenza tasso SI (≥1 mancante − nessuno): {gruppi_miss.tasso_SI_pct.iloc[0] - gruppi_miss.tasso_SI_pct.iloc[1]:+.2f} punti; "
      f"chi2={chi2:.2f} (manuale {chi2_manuale:.2f}), p={p_chi2:.4f}; Fisher p={p_fisher:.4f}")

# (c) "effetto dose": tasso di Hiring per numero di skill mancanti
classe = pd.cut(k_miss, [-1, 0, 1, 2, 16], labels=["0", "1", "2", "3+"])
dose = pd.DataFrame({k: tasso(classe == k) for k in classe.cat.categories}).T
display(dose)

,n_mancanti,pct_mancanti,n_totale
Entrepreneurial Mindset,153,3.49,4382
Critical Thinking,127,2.90,4382
Creativity,96,2.19,4382
Planning,95,2.17,4382
Teamwork,90,2.05,4382
Digital,65,1.48,4382
Cultural Awareness,49,1.12,4382
Time Management,22,0.50,4382
Interpersonal Skills,20,0.46,4382
Flexibility,12,0.27,4382


Righe per numero di skill mancanti: {0: 3912, 1: 295, 2: 110, 3: 43, 4: 13, 5: 1, 6: 3, 7: 2, 9: 1, 15: 2}


,n,n_SI,tasso_SI_pct,IC95_basso,IC95_alto
≥1 skill mancante,470.0,232.0,49.36,44.9,53.9
nessuna skill mancante,3909.0,2262.0,57.87,56.3,59.4


Differenza tasso SI (≥1 mancante − nessuno): -8.51 punti; chi2=12.38 (manuale 12.38), p=0.0004; Fisher p=0.0005


,n,n_SI,tasso_SI_pct,IC95_basso,IC95_alto
0,3909.0,2262.0,57.87,56.3,59.4
1,295.0,160.0,54.24,48.5,59.8
2,110.0,43.0,39.09,30.5,48.4
3+,65.0,29.0,44.62,33.2,56.7


In [20]:
# (d) per singola skill: tasso di Hiring quando la skill è mancante vs presente (correzione di Holm)
righe = []
for c in SKILLS:
    mc = m_na[c]
    if mc.sum() == 0:
        continue
    a, b = tasso(mc), tasso(~mc)
    p = stats.fisher_exact([[a["n_SI"], a["n"] - a["n_SI"]], [b["n_SI"], b["n"] - b["n_SI"]]])[1]
    righe.append({"skill": c, "n_mancante": a["n"], "tasso_SI_mancante": a["tasso_SI_pct"],
                  "n_presente": b["n"], "tasso_SI_presente": b["tasso_SI_pct"],
                  "diff_punti": round(a["tasso_SI_pct"] - b["tasso_SI_pct"], 2), "p_fisher": p})
hiring_per_skill = pd.DataFrame(righe)
hiring_per_skill["p_holm"] = multipletests(hiring_per_skill.p_fisher, method="holm")[1]
hiring_per_skill[["p_fisher", "p_holm"]] = hiring_per_skill[["p_fisher", "p_holm"]].round(4)
hiring_per_skill = hiring_per_skill.sort_values("diff_punti")
hiring_per_skill.to_csv(OUT_TAB / "step1_mancanti_vs_hiring_per_skill.csv", index=False)
display(hiring_per_skill)

# (e) OVERALL e DURATA: chi ha mancanti ha giudizi o durate diverse? (Mann-Whitney, voti ordinali)
ov = num["OVERALL"].where(m_valido["OVERALL"])
du = dur.where(~m_excel)
righe = []
for nome, v in [("OVERALL", ov), ("DURATA (senza date Excel)", du)]:
    g1, g0 = v[(k_miss > 0) & v.notna()], v[(k_miss == 0) & v.notna()]
    righe.append({"variabile": nome, "n_con_mancanti": len(g1), "media_con": round(g1.mean(), 2), "mediana_con": g1.median(),
                  "n_senza": len(g0), "media_senza": round(g0.mean(), 2), "mediana_senza": g0.median(),
                  "p_mann_whitney": round(stats.mannwhitneyu(g1, g0).pvalue, 4)})
display(pd.DataFrame(righe))

# (f) segmenti: la quota di righe con mancanti cambia tra paesi, aree, corsi? (solo categorie con n>=30)
righe = []
seg_naz = np.where(naz.eq("ITALIA"), "ITALIA", np.where(naz.eq(""), "", "ESTERO"))
for nome, s in [("NAZIONE (Italia/estero)", pd.Series(seg_naz, index=df_raw.index)),
                ("AREA FUNZIONALE (Legale raggruppato)", area_gr), ("CORSO", co)]:
    vc = s[s != ""].value_counts(); tenute = vc[vc >= SOGLIA_N].index
    m = s.isin(tenute)
    ct = pd.crosstab(s[m], k_miss[m] > 0)
    quote = (100 * ct[True] / ct.sum(axis=1)).round(1)
    righe.append({"segmento": nome, "n_righe": int(m.sum()), "n_categorie": len(tenute),
                  "quota_min_pct": quote.min(), "categoria_min": quote.idxmin(),
                  "quota_max_pct": quote.max(), "categoria_max": quote.idxmax(),
                  "p_chi2": round(stats.chi2_contingency(ct)[1], 4)})
mancanti_segmenti = pd.DataFrame(righe)
mancanti_segmenti.to_csv(OUT_TAB / "step1_mancanti_per_segmento.csv", index=False)
display(mancanti_segmenti)

# (g) Controllo aggiustato: il legame mancanti–Hiring resta a parità di area e corso?
#     (regressione logistica; solo aree e corsi con n>=30; linguaggio di associazione, non causale)
import statsmodels.formula.api as smf
d = pd.DataFrame({"hiring": y.astype(int), "mancante": (k_miss > 0).astype(int),
                  "area": area_gr, "corso": co})[camp]
vc_a, vc_c = d.area.value_counts(), d.corso.value_counts()
d = d[d.area.isin(vc_a[vc_a >= SOGLIA_N].index) & d.corso.isin(vc_c[vc_c >= SOGLIA_N].index)]
righe = []
for nome, formula in [("non aggiustato", "hiring ~ mancante"),
                      ("aggiustato per area e corso", "hiring ~ mancante + C(area) + C(corso)")]:
    m = smf.logit(formula, d).fit(disp=0)
    lo, hi = np.exp(m.conf_int().loc["mancante"])
    righe.append({"modello": nome, "n": int(m.nobs), "OR_mancante": round(np.exp(m.params["mancante"]), 3),
                  "IC95_basso": round(lo, 3), "IC95_alto": round(hi, 3), "p": round(m.pvalues["mancante"], 4)})
logit_mancanti = pd.DataFrame(righe)
# Verifica indipendente: l'OR non aggiustato coincide con l'odds ratio della tabella 2x2 sullo stesso campione
t = pd.crosstab(d.mancante, d.hiring)
or_manuale = (t.loc[1, 1] * t.loc[0, 0]) / (t.loc[1, 0] * t.loc[0, 1])
assert np.isclose(or_manuale, logit_mancanti.OR_mancante.iloc[0], atol=5e-4), (or_manuale, logit_mancanti)
logit_mancanti.to_csv(OUT_TAB / "step1_mancanti_vs_hiring_logit.csv", index=False)
display(logit_mancanti)
print(f"OR manuale (2x2) = {or_manuale:.3f}")

registra("Mancanti non casuali: chi ha N/A ha meno Hiring, OVERALL più basso, stage più brevi; quota diversa per area e corso",
         "15 skill con N/A", r_na,
         "Nessuna imputazione; medie per skill sui valori disponibili con n riportato; confronto di sensibilità sulle sole righe complete; dichiarare il limite",
         "Le medie calcolate sui soli valori presenti possono risultare leggermente ottimistiche, soprattutto per le skill con più N/A",
         "DA DECIDERE")

,skill,n_mancante,tasso_SI_mancante,n_presente,tasso_SI_presente,diff_punti,p_fisher,p_holm
12,Motivation,12,25.00,4367,57.04,-32.04,0.0375,0.4496
11,Responsibility,11,27.27,4368,57.03,-29.76,0.0652,0.7170
1,Teamwork,90,38.89,4289,57.33,-18.44,0.0005,0.0075
3,Critical Thinking,127,40.94,4252,57.43,-16.49,0.0003,0.0038
13,Feedback Management,12,41.67,4367,57.00,-15.33,0.3830,1.0000
2,Entrepreneurial Mindset,153,44.44,4226,57.41,-12.97,0.0020,0.0261
14,Interpersonal Skills,20,45.00,4359,57.01,-12.01,0.3658,1.0000
4,Creativity,96,47.92,4283,57.16,-9.24,0.0766,0.7659
10,Time Management,22,54.55,4357,56.97,-2.42,0.8321,1.0000
7,Cultural Awareness,49,55.10,4330,56.97,-1.87,0.8848,1.0000


,variabile,n_con_mancanti,media_con,mediana_con,n_senza,media_senza,mediana_senza,p_mann_whitney
0,OVERALL,470,9.00,9.0,3910,9.18,10.0,0.0002
1,DURATA (senza date Excel),470,87.76,77.0,3905,109.84,91.0,0.0000


,segmento,n_righe,n_categorie,quota_min_pct,categoria_min,quota_max_pct,categoria_max,p_chi2
0,NAZIONE (Italia/estero),4379,2,10.6,ITALIA,11.1,ESTERO,0.7474
1,AREA FUNZIONALE (Legale raggruppato),4336,12,5.6,"Produzione, logistica e qualità",22.9,Ricerca & Sviluppo,0.0000
2,CORSO,4286,24,4.3,B10,21.7,B09,0.0000


,modello,n,OR_mancante,IC95_basso,IC95_alto,p
0,non aggiustato,4243,0.699,0.576,0.850,0.0003
1,aggiustato per area e corso,4243,0.729,0.595,0.894,0.0024


OR manuale (2x2) = 0.699


### 1.7 Coerenza tra OVERALL e media delle skill

In [21]:
sk = num[SKILLS].where(m_valido[SKILLS])            # solo voti validi
n_sk = sk.notna().sum(axis=1)
media_sk = sk.mean(axis=1)                          # media delle skill disponibili
scarto = ov - media_sk                              # OVERALL − media skill
piatta16 = (n_sk == 16) & (sk.nunique(axis=1) == 1) # usato qui per la robustezza, definito in 1.8

def coerenza(mask, nome):
    x, z = media_sk[mask], ov[mask]
    d = (z - x)
    return {"campione": nome, "n": int(mask.sum()),
            "spearman_rho": round(stats.spearmanr(x, z).statistic, 3),
            "pearson_r": round(stats.pearsonr(x, z).statistic, 3),
            "scarto_medio": round(d.mean(), 3), "scarto_mediano": round(d.median(), 3),
            "pct_abs_scarto_ge_1": pct((d.abs() >= 1).sum(), len(d)),
            "pct_abs_scarto_ge_2": pct((d.abs() >= 2).sum(), len(d)),
            "n_abs_scarto_ge_2": int((d.abs() >= 2).sum()),
            "n_overall_fuori_min_max_skill": int(((z < sk[mask].min(axis=1)) | (z > sk[mask].max(axis=1))).sum())}

base      = ov.notna() & (n_sk >= 1)
completi  = ov.notna() & (n_sk == 16)
coer = pd.DataFrame([coerenza(base, "OVERALL valido e ≥1 skill valida"),
                     coerenza(completi, "solo righe con 16 skill valide"),
                     coerenza(completi & ~piatta16, "16 skill valide, escluse le piatte")])
coer.to_csv(OUT_TAB / "step1_overall_vs_media_skill.csv", index=False)
display(coer)

# Verifica indipendente: Pearson con la formula manuale
x, z = media_sk[base].to_numpy(), ov[base].to_numpy()
r_man = ((x - x.mean()) * (z - z.mean())).sum() / np.sqrt(((x - x.mean())**2).sum() * ((z - z.mean())**2).sum())
assert np.isclose(r_man, coer.pearson_r.iloc[0], atol=5e-4), (r_man, coer.pearson_r.iloc[0])
print(f"Pearson manuale = {r_man:.4f}")

# Distribuzione dello scarto (campione base): classi definite con condizioni esplicite
sc = scarto[base]
condizioni = [sc <= -2, (sc > -2) & (sc <= -1), (sc > -1) & (sc < 1), (sc >= 1) & (sc < 2), sc >= 2]
etichette  = ["scarto ≤ −2", "−2 < scarto ≤ −1", "−1 < scarto < +1", "+1 ≤ scarto < +2", "scarto ≥ +2"]
distr_scarto = pd.DataFrame({"n": [int(c.sum()) for c in condizioni]}, index=etichette)
distr_scarto["pct"] = (100 * distr_scarto.n / len(sc)).round(2)
assert distr_scarto.n.sum() == len(sc)
# coerenza con le soglie della tabella precedente
assert distr_scarto.n.iloc[[0, 4]].sum() == coer.n_abs_scarto_ge_2.iloc[0]
assert np.isclose(pct(distr_scarto.n.drop("−1 < scarto < +1").sum(), len(sc)), coer.pct_abs_scarto_ge_1.iloc[0])
display(distr_scarto)

discordanti = df_raw.loc[base & (scarto.abs() >= 2), ["ID", "OVERALL", "Hiring"]].assign(
    media_skill=media_sk.round(2), n_skill_valide=n_sk, scarto=scarto.round(2)).sort_values("scarto")
discordanti.to_csv(OUT_TAB / "step1_overall_discordanti.csv", index=False)

registra("OVERALL distante dalla media delle skill (|scarto| ≥ 2)", "OVERALL, 16 skill", base & (scarto.abs() >= 2),
         "Nessuna correzione (OVERALL è un giudizio autonomo del tutor); segnalare e verificare la robustezza",
         "Possibili errori di compilazione non individuabili", "DA DECIDERE")

,campione,n,spearman_rho,pearson_r,scarto_medio,scarto_mediano,pct_abs_scarto_ge_1,pct_abs_scarto_ge_2,n_abs_scarto_ge_2,n_overall_fuori_min_max_skill
0,OVERALL valido e ≥1 skill valida,4380,0.862,0.919,0.099,0.062,4.27,0.21,9,10
1,solo righe con 16 skill valide,3904,0.859,0.920,0.106,0.062,4.20,0.18,7,7
2,"16 skill valide, escluse le piatte",3050,0.852,0.908,0.136,0.188,5.28,0.23,7,4


Pearson manuale = 0.9193


,n,pct
scarto ≤ −2,8,0.18
−2 < scarto ≤ −1,61,1.39
−1 < scarto < +1,4193,95.73
+1 ≤ scarto < +2,117,2.67
scarto ≥ +2,1,0.02


### 1.8 Valutazioni piatte (stesso voto su tutte le skill)

In [22]:
n_distinti = sk.nunique(axis=1)
piatta_parz = (n_sk.between(1, 15)) & (n_distinti == 1)      # stesso voto su tutte le skill presenti, con N/A
valore_piatto = sk.max(axis=1)

# Verifica indipendente (stessa logica di una formula Excel): 16 numeri e MIN = MAX sui valori grezzi
excel_like = (num[SKILLS].notna().sum(axis=1) == 16) & (num[SKILLS].min(axis=1) == num[SKILLS].max(axis=1))
assert (excel_like == piatta16).all()

piatte = pd.DataFrame({
    "n": [piatta16.sum(), piatta_parz.sum()],
    "pct_sul_totale": [pct(piatta16.sum()), pct(piatta_parz.sum())],
    "pct_su_righe_16_skill_valide": [pct(piatta16.sum(), (n_sk == 16).sum()), np.nan],
}, index=["piatta: 16 skill valide e identiche", "piatta parziale: identiche ma con ≥1 N/A"])
display(piatte)
print("Righe con 16 skill valide:", int((n_sk == 16).sum()))

per_valore = valore_piatto[piatta16].value_counts().sort_index(ascending=False).to_frame("n")
per_valore["pct_delle_piatte"] = (100 * per_valore.n / piatta16.sum()).round(2)
per_valore["n_con_OVERALL_uguale"] = [int((piatta16 & (valore_piatto == v) & (ov == v)).sum()) for v in per_valore.index]
display(per_valore)

print("Numero di voti distinti sulle 16 skill (righe complete):",
      n_distinti[n_sk == 16].value_counts().sort_index().to_dict())
print(f"OVERALL medio: tutte n={int(ov.notna().sum())} -> {ov.mean():.3f}; "
      f"escluse le piatte n={int((ov.notna() & ~piatta16).sum())} -> {ov[~piatta16].mean():.3f}")
pd.concat([per_valore], keys=["piatta16"]).to_csv(OUT_TAB / "step1_valutazioni_piatte.csv")

registra("Valutazioni piatte (16 skill valide e identiche)", "16 skill", piatta16,
         "Tenere (non sono errori dimostrabili); flag e analisi di sensibilità con/senza",
         "Escluderle toglierebbe soprattutto giudizi 'tutto 10' e abbasserebbe le medie; tenerle comprime le differenze tra skill",
         "DA DECIDERE")

,n,pct_sul_totale,pct_su_righe_16_skill_valide
piatta: 16 skill valide e identiche,854,19.49,21.87
piatta parziale: identiche ma con ≥1 N/A,87,1.99,NaN


Righe con 16 skill valide: 3905


,n,pct_delle_piatte,n_con_OVERALL_uguale
10.0,743,87.00,742
9.0,64,7.49,63
8.0,33,3.86,32
7.0,8,0.94,8
6.0,5,0.59,5
5.0,1,0.12,1


Numero di voti distinti sulle 16 skill (righe complete): {1: 854, 2: 1194, 3: 1197, 4: 507, 5: 120, 6: 25, 7: 7, 8: 1}
OVERALL medio: tutte n=4380 -> 9.163; escluse le piatte n=3526 -> 9.011


### 1.9 Tabella riassuntiva dell'audit

In [23]:
riepilogo = pd.DataFrame(audit)
riepilogo.to_csv(OUT_TAB / "step1_audit_riepilogo.csv", index=False)
with pd.option_context("display.max_colwidth", 120):
    display(riepilogo)

# Controllo finale: nessun trattamento applicato al dataset grezzo
assert int(pd.util.hash_pandas_object(df_raw, index=True).sum()) == FINGERPRINT_RAW
assert df_raw.shape == (N_ROWS_RAW, 24)
print("df_raw invariato: nessun trattamento applicato.")

,problema,colonne,n_righe,pct_sul_totale,trattamento_proposto,rischio_del_trattamento,decisione
0,Testo non numerico nei voti,"Cultural Awareness, Listening, Interpersonal Skills",3,0.07,Convertire in mancante solo per quella skill,Minimo: 3 celle; il voto reale è irrecuperabile,confermato
1,"Fuori scala ricodificabile come refuso (55, 88, 999, 100)","Cultural Awareness, Writing, Responsibility, OVERALL",4,0.09,"Ricodificare in 5, 8, 9, 10",Il valore reale potrebbe essere diverso (es. 999 è spesso un codice di 'mancante'),confermato
2,Fuori scala non ricodificabile (7000),Entrepreneurial Mindset,1,0.02,Convertire in mancante solo per quella skill,Minimo: 1 cella,confermato
3,Mancante dichiarato (N/A),"15 skill su 16 (non Leadership, non OVERALL)",470,10.73,"Mancante solo per quella skill; nessuna esclusione di riga, nessuna imputazione",Se i mancanti non sono casuali (1.6) le medie per skill possono essere distorte; n diverso tra skill,confermato
4,Riga quasi vuota,"tutte tranne ID, DURATA, SETTORE",1,0.02,Escludere la riga,Nessuno rilevante: la riga non contiene valutazioni,confermato
5,ID duplicato,ID,0,0.00,Nessuno,-,nessun caso
6,Righe identiche su tutte le colonne tranne ID,tutte tranne ID,18,0.41,Nessuno: ID diverso = stage diverso,"Se fossero doppi inserimenti, 9 valutazioni (quasi tutte 'tutto 10') conterebbero due volte",confermato
7,SETTORE AZIENDALE senza struttura plausibile (distribuzione uniforme p=0.85; nessuna coerenza con l'area),SETTORE AZIENDALE,4382,100.00,"Non usarlo come dimensione di segmentazione; al più descrittivo, dichiarando il motivo","Se il dato fosse corretto, si rinuncerebbe a una dimensione di lettura",DA DECIDERE
8,NAZIONE mancante,NAZIONE,3,0.07,Mancante solo per le analisi per paese,Minimo: 3 righe (una è la riga quasi vuota),confermato
9,AREA FUNZIONALE o CORSO mancante,"AREA FUNZIONALE, CORSO",1,0.02,Mancante solo per quella variabile,Nessuno: è la riga quasi vuota,confermato


df_raw invariato: nessun trattamento applicato.
